Swarm implements a team in which agents can hand off task to other agents based on their capabilities. It is a multi-agent design pattern first introduced by OpenAI in Swarm. The key idea is to let agent delegate tasks to other agents using a special tool call, while all agents share the same message context. This enables agents to make local decisions about task planning, rather than relying on a central orchestrator such as in SelectorGroupChat.

The overall process can be summarized as follows:

1. Each agent has the ability to generate HandoffMessage to signal which other agents it can hand off to. For AssistantAgent, this means setting the handoffs argument.

2. When the team starts on a task, the first speaker agents operate on the task and make localized decision about whether to hand off and to whom.

3. When an agent generates a HandoffMessage, the receiving agent takes over the task with the same message context.

4. The process continues until a termination condition is met.

In [1]:
from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="gemini-2.5-flash",
    api_key=api_key,
)

In [3]:
def refund_flight(flight_PNR:str) ->str:
    return f"Refunded Flight with PNR {flight_PNR}"

In [4]:
travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

In [5]:
flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight PNR numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)

In [6]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)

In [ ]:
await team.reset()

In [ ]:
task = ' I want to refund my flight'

async def run_team_stream() -> None:

    task_result = await Console(team.run_stream(task = task))

    last_message = task_result.messages[-1]


    while ( isinstance(last_message,HandoffMessage) and last_message.target == 'user'):

        user_ka_message = input("User : ")

        task_result = await Console(team.run_stream(task = HandoffMessage(source='user',target=last_message.source,content=user_ka_message)))

        last_message = task_result.messages[-1]

await run_team_stream()

In [9]:
flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight PNR numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)

In [10]:
await team.reset()

In [11]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE") 
team = Swarm([flights_refunder,travel_agent ], termination_condition=termination,max_turns=20)

In [ ]:

termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE") 
team = Swarm([flights_refunder,travel_agent ], termination_condition=termination,max_turns=20)
task = ' I want to refund my flight'

async def run_team_stream() -> None:

    task_result = await Console(team.run_stream(task = task))

    last_message = task_result.messages[-1]


    while ( isinstance(last_message,HandoffMessage) and last_message.target == 'user'):

        user_ka_message = input("User : ")

        task_result = await Console(team.run_stream(task = HandoffMessage(source='user',target=last_message.source,content=user_ka_message)))

        last_message = task_result.messages[-1]

await run_team_stream()